<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c39-header.png" alt="Nextia Learning · Entity Resolution and Record Matching: Is This the Same Thing?" width="100%">

# Find duplicate people and companies

**[Find duplicate people and companies](https://learning.nextia-ai.com/courses/record-matching/m09/find-duplicate-people-and-companies/)** · Entity Resolution and Record Matching: Is This the Same Thing? · Module 9, case study 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** find duplicates in one list, from start to end, twice: simulated people, then real companies from the global LEI register. You will compare rules, Splink without labels and classifiers under one measure, cluster the pairs into entities, build golden records, and choose a design with its thresholds and its review queue.

| | |
|---|---|
| **Time** | About 60 minutes in the notebook (the lesson page has the full story) |
| **Needs** | An internet connection for the setup cells (about 2 MB) |
| **You should know** | The course's steps, from [A supervised pair classifier](https://learning.nextia-ai.com/courses/record-matching/m06/a-supervised-pair-classifier/) and [The golden record](https://learning.nextia-ai.com/courses/record-matching/m07/the-golden-record/) |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, rapidfuzz 3.14.6, scikit-learn 1.9.1, LightGBM 4.7.0, networkx 3.7, Splink 5.0.0 and DuckDB 2.0.0, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/record-matching/m09/find-duplicate-people-and-companies/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/M09-L03-find-duplicate-people-and-companies/find-duplicate-people-and-companies-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, rapidfuzz 3.14.6, scikit-learn 1.9.1, lightgbm 4.7.0, networkx 3.7, splink 5.0.0, duckdb 2.0.0, pyarrow 26.0.1, and matplotlib. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 rapidfuzz==3.14.6 scikit-learn==1.9.1 lightgbm==4.7.0 networkx==3.7 splink==5.0.0 duckdb==2.0.0 pyarrow==26.0.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'rapidfuzz', 'scikit-learn', 'lightgbm', 'networkx', 'splink', 'duckdb', 'pyarrow']:
    print(name, version(name))

## Setup: get the data

Run the next cell. It downloads the simulated people (56 KB) from Splink's datasets repository, and the GLEIF slice (about 2 MB) from the course's labs repository, into a folder `duplicates`. It checks every file's checksum. It needs an internet connection and no account.

You should see `Ready:` and the files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `duplicates`, then run the cell again. If it happens again, the file at the source has changed: the notebook stops, so that it never runs on other data than the lesson.

**Sources and licences.** People: `fake_1000.csv` from [moj-analytical-services/splink_datasets](https://github.com/moj-analytical-services/splink_datasets) (MIT licence, © 2023 MoJ Analytical Services), simulated records of invented people. Companies: LEI records from the [GLEIF API](https://api.gleif.org/api/v1/lei-records), filtered to legal addresses in Ireland, under [CC0 1.0](https://creativecommons.org/publicdomain/zero/1.0/) ([GLEIF terms of use](https://www.gleif.org/en/meta/lei-data-terms-of-use)); Nextia Learning kept 15 columns and wrote them as one file. See `dataset.md`.

In [ ]:
# Setup: download the simulated people from their source and the GLEIF slice from labs; check every checksum.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

SOURCES = {  # the simulated people in Splink's datasets repository (one fixed version), and its SHA-256
    "data/fake_1000.csv": ("https://raw.githubusercontent.com/moj-analytical-services/splink_datasets/75876d806d9eff72072d21878150ee50a96d5f41/data/fake_1000.csv", "66f9b8f92b9b082ef982fe18c77030fbed8780f06ce1d40d7106be5a72392e2a"),
}
LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C39/M09-L03-find-duplicate-people-and-companies/"
FILES = {
    "data/gleif-ie.csv.gz": "73f05300cd9be87a06953e04aa1006ee14d69720e094a492ee2285955ba67c2c",
    "data/elf-codes.csv": "af2960f554e193a741e1a0a78bc220018112ee2f9260362cb6b786735380fbb4"
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("duplicates").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, expected):
    path = WORK / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        shutil.copy(local, path) if local.exists() else download(url, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder duplicates and run again.")
    return path

for name, (url, expected) in SOURCES.items():
    fetch(name, url, expected)
for name, expected in FILES.items():
    fetch(name, LABS + name, expected)
os.chdir(WORK)
print("Ready:", sorted(p.name for p in Path("data").iterdir()))

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd` and matplotlib. It defines:

- `text()` normalises text; `split_entities()` puts every entity in train, validation or test; `pair_split()` gives the split of a pair.
- `pair_measures()` gives pair precision, recall and F1; `best_threshold()` chooses a threshold on validation; `band_table()` counts the accept and review bands.
- `components()` turns links into clusters; `cluster_measures()` gives B-cubed and exact-cluster precision and recall.
- `splink_connection()` opens DuckDB with a cap on memory and temporary files.
- The check helpers `expect()` and `expect_hash()`: they say if your answer is right, without showing the answer.

Run the cell. It prints nothing.

In [ ]:
# Helpers: normalise, split by entity, measure pairs and clusters, and checks that do not show the answer.
import hashlib, json, re, time, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 39

def text(s):
    """Lower case, no accents, signs to spaces, one space between words (the course's normalise.text)."""
    s = "".join(c for c in unicodedata.normalize("NFKD", str(s or "")) if not unicodedata.combining(c)).lower()
    s = re.sub(r"[^\w\s]", " ", s.replace("'", "").replace("\u2019", ""))
    return re.sub(r"\s+", " ", s).strip()

def split_entities(entities, shares=(0.5, 0.25, 0.25), seed=SEED):
    """train / validation / test for every entity: all records of one entity stay in one split."""
    groups = sorted(set(entities))
    order = np.random.default_rng(seed).permutation(len(groups))
    cut1, cut2 = int(shares[0] * len(groups)), int((shares[0] + shares[1]) * len(groups))
    names = np.empty(len(groups), dtype=object)
    names[order[:cut1]], names[order[cut1:cut2]], names[order[cut2:]] = "train", "validation", "test"
    return dict(zip(groups, names))

def pair_measures(called, pairs, which="test"):
    """Pair precision, recall and F1 on one split. `pairs` has the columns `same` and `split`; `called` is a
    True/False array: the pairs the matcher calls the same. A split has every true pair of its entities."""
    called = np.asarray(called, dtype=bool)
    in_split = (pairs.split == which).to_numpy()
    true = int(pairs.attrs["true_pairs"][which])
    c = called & in_split
    right = int((c & pairs.same.to_numpy()).sum())
    p = right / c.sum() if c.sum() else 0.0
    r = right / true if true else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return {"precision": round(float(p), 3), "recall": round(float(r), 3), "f1": round(float(f1), 3), "called": int(c.sum()),
            "right": right, "true": true}

def pair_split(left, right):
    """The split of a pair: test if one of its records is in test, else validation if one is, else train.
    So a train pair has two train records, and a wrong link from a test record to any other record counts."""
    l, r = np.asarray(left, dtype=object), np.asarray(right, dtype=object)
    return np.where((l == "test") | (r == "test"), "test",
                    np.where((l == "validation") | (r == "validation"), "validation", "train"))

def best_threshold(score, pairs):
    """The threshold with the best F1 on the validation split (the course's rule)."""
    s = np.asarray(score, dtype=float)
    cands = np.unique(np.round(s[(pairs.split == "validation").to_numpy()], 4))
    best = max(cands, key=lambda t: (pair_measures(s >= t, pairs, "validation")["f1"], -t))
    return float(best)

def components(ids, left, right):
    """Connected components (Module 7): every chain of links becomes one cluster. Returns {id: cluster}."""
    parent = {x: x for x in ids}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in zip(left, right):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    return {x: find(x) for x in ids}

def cluster_measures(records, cluster, which="test"):
    """B-cubed and exact-cluster precision and recall (Module 7) on the records of one split's entities."""
    df = records.assign(cluster=records.id.map(cluster))
    c_size = df.groupby("cluster").id.transform("size")
    e_size = df.groupby("entity").id.transform("size")
    both = df.groupby(["cluster", "entity"]).id.transform("size")
    keep = (df.split == which).to_numpy()
    cl = df.groupby("cluster").id.apply(frozenset)
    en = df.groupby("entity").id.apply(frozenset)
    cl_in = cl[cl.index.isin(df.cluster[keep])]
    en_in = en[en.index.isin(df.entity[keep])]
    exact = len(set(cl_in) & set(en_in))
    return {"bcubed precision": round(float((both / c_size)[keep].mean()), 3),
            "bcubed recall": round(float((both / e_size)[keep].mean()), 3),
            "exact precision": round(exact / len(cl_in), 3), "exact recall": round(exact / len(en_in), 3),
            "entities found": int(len(cl_in)), "true entities": int(len(en_in)),
            "largest": int(df[keep].groupby("cluster").size().max())}

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def splink_connection():
    """A DuckDB connection for Splink: one thread, 2 GB of memory, temporary files in one folder, at most 4 GB."""
    import duckdb
    Path("duckdb_tmp").mkdir(exist_ok=True)
    con = duckdb.connect()
    con.execute("SET threads = 1; SET memory_limit = '2GB'; SET temp_directory = 'duckdb_tmp'; "
                "SET max_temp_directory_size = '4GB'")
    return con

def band_table(score, pairs, which, uppers, lower):
    """Accept at `upper` or more, review from `lower` to `upper`: the counts on one split, for each upper."""
    rows = {}
    s = np.asarray(score)
    for up in uppers:
        ins = (pairs.split == which).to_numpy()
        acc, rev = ins & (s >= up), ins & (s >= lower) & (s < up)
        rows[up] = {"accepted": int(acc.sum()), "accepted right": int((acc & pairs.same).sum()),
                    "to review": int(rev.sum()), "true in review": int((rev & pairs.same).sum()),
                    "recall if reviews are right": round(float(((acc | rev) & pairs.same).sum() / pairs.attrs["true_pairs"][which]), 3)}
    t = pd.DataFrame(rows).T
    t.insert(2, "precision accepted", (t["accepted right"] / t.accepted.where(t.accepted > 0)).round(3))
    return t

results = {}   # the comparison table: one row per method

def add_result(table, name, called, pairs, threshold=None):
    row = pair_measures(called, pairs, "test")
    results.setdefault(table, {})[name] = {**row, "threshold": threshold}
    return row

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## Part 1. People

## 1. The situation

A charity, a club or a clinic keeps one list of people. Over the years, the same person signs up two or three times: with a nickname, a typing error, an old email, or the first and last names swapped. The team wants **one record per person**. A **false merge** joins two different people, so one person can see another person's details. A **missed duplicate** sends two letters to one person.

Real people's records cannot be shared in a public course. So part 1 uses **simulated people**: Splink's `fake_1000` list, made by a program from invented people, with the true person of every record.

In [ ]:
raw = pd.read_csv("data/fake_1000.csv", dtype=str, keep_default_na=False)
print(raw.shape, raw.columns.tolist())
print(raw.head(8).to_string(index=False))

> **Check.** 1,000 records and 7 columns. `cluster` is the true person: a label, never an input. Records 0 to 3 are one person, written four ways.

In [ ]:
print("repeated IDs:", int(raw.unique_id.duplicated().sum()))
print("empty values:", {c: int((raw[c] == "").sum()) for c in ["first_name", "surname", "dob", "city", "email"]})
sizes = raw.cluster.value_counts()
print("people:", raw.cluster.nunique(), "| records per person:", sizes.value_counts().sort_index().to_dict())

> **Check.** No repeated IDs. Many values are empty (about 1 in 5 of the names, cities and emails). 251 people have 1 to 7 records each.

## 2. Split, pairs and blocking

You split by **person**: all records of one person go to train, validation or test. A pair belongs to test if one of its records is in test, so a wrong link from a test person to anybody counts as an error. 1,000 records make 499,500 pairs: few enough to compare every pair on a laptop.

In [ ]:
people = pd.DataFrame({
    "id": raw.unique_id.astype(int), "first": raw.first_name.map(text), "last": raw.surname.map(text),
    "dob": raw.dob, "city": raw.city.map(text), "email": raw.email.str.strip().str.lower(),
    "entity": raw.cluster.astype(int)})                     # `entity` is the label: never an input
p_split = split_entities(people.entity)
people["split"] = people.entity.map(p_split)

# All pairs: 1,000 records make 499,500 pairs, few enough to compare every one.
i, j = np.triu_indices(len(people), k=1)
L, R = people.iloc[i].reset_index(drop=True), people.iloc[j].reset_index(drop=True)
p_pairs = pd.DataFrame({"id_l": L.id, "id_r": R.id, "same": (L.entity == R.entity).to_numpy(),
                        "split": pair_split(L.split, R.split)})
p_pairs.attrs["true_pairs"] = p_pairs[p_pairs.same].split.value_counts().to_dict()
print(f"{len(p_pairs):,} pairs;", "true pairs by split:", p_pairs.attrs["true_pairs"])

> **Predict.** Module 3's blocking keys kept 99% of Larkfield's true pairs. What share do you expect here with four keys (date of birth, email, two letters of each name, three letters of the surname)?

In [ ]:
def key_pairs(key):
    """The pairs that share a non-empty value of one key (a blocking key, Module 3)."""
    ok = (key != "").to_numpy()
    return (key.iloc[i].to_numpy() == key.iloc[j].to_numpy()) & ok[i] & ok[j]

keys = {"date of birth": people.dob,
        "email": people.email,
        "2 letters of first + last name": (people["first"].str[:2] + people["last"].str[:2]).where(
            (people["first"] != "") & (people["last"] != ""), ""),
        "first 3 letters of last name": people["last"].str[:3]}
rows, union = {}, np.zeros(len(p_pairs), bool)
for name, key in keys.items():
    kp = key_pairs(key)
    union |= kp
    rows[name] = {"candidates": int(kp.sum()), "true kept": int((kp & p_pairs.same).sum())}
rows["all four together"] = {"candidates": int(union.sum()), "true kept": int((union & p_pairs.same).sum())}
p_blocking = pd.DataFrame(rows).T
p_blocking["pair completeness"] = (p_blocking["true kept"] / p_pairs.same.sum()).round(3)
print(p_blocking.to_string())

> **Check.** Together, the four keys keep only about 79% of the true pairs. These records have many more errors than Larkfield's. With 499,500 pairs, you can skip blocking and compare all pairs. With a million records you could not: you would need looser keys.

## 3. Baselines

The simplest matchers: the same name and date of birth, or the same email.

In [ ]:
full = (L["first"] != "") & (L["last"] != "") & (R["first"] != "") & (R["last"] != "")
exact = (full & (L["first"] == R["first"]) & (L["last"] == R["last"]) & (L.dob == R.dob)).to_numpy()
print("exact name and date of birth:", add_result("people", "exact name + date of birth", exact, p_pairs))
same_email = ((L.email != "") & (L.email == R.email)).to_numpy()
print("same email:", add_result("people", "rule: same email", same_email, p_pairs))

> **Check.** Both are always right (precision 1.0), but the exact rule finds only 25 of the 498 test pairs.

## 4. Compare

Each pair gets a comparison vector of seven similarities (Module 4). `swapped_jw` compares the first name with the other record's last name, for records where someone swapped the two. A missing value stays missing.

In [ ]:
from rapidfuzz.distance import JaroWinkler, Levenshtein

def sim(a, b, scorer=JaroWinkler.normalized_similarity):
    """One similarity per pair; missing (NaN) when one side is empty."""
    return np.array([scorer(x, y) if x and y else np.nan for x, y in zip(a, b)])

t0 = time.perf_counter()
p_vec = pd.DataFrame({
    "first_jw": sim(L["first"], R["first"]),
    "last_jw": sim(L["last"], R["last"]),
    "swapped_jw": np.fmax(sim(L["first"], R["last"]), sim(L["last"], R["first"])),   # first and last swapped
    "dob_lev": sim(L.dob, R.dob, Levenshtein.normalized_similarity),                  # typing errors in the date
    "dob_log_days": np.log1p((pd.to_datetime(L.dob) - pd.to_datetime(R.dob)).dt.days.abs().to_numpy()),
    "city_jw": sim(L.city, R.city),
    "email_jw": sim(L.email, R.email),
})
p_compare_seconds = time.perf_counter() - t0
print(f"{len(p_vec):,} comparison vectors in {p_compare_seconds:.1f} s")
print(p_vec.head(3).round(3).to_string())

## 5. Splink without labels, and two classifiers

Splink learns its weights without labels (Module 5). Its settings are close to Splink's own tutorial on this list.

In [ ]:
import logging
import splink.comparison_library as cl
from splink import Linker, SettingsCreator, block_on
from splink.backends.duckdb import DuckDBAPI

p_settings = SettingsCreator(
    link_type="dedupe_only", unique_id_column_name="unique_id",
    comparisons=[cl.JaroWinklerAtThresholds("first_name", [0.92, 0.8]),
                 cl.JaroWinklerAtThresholds("surname", [0.92, 0.8]),
                 cl.DateOfBirthComparison("dob", input_is_string=True, datetime_format="%Y-%m-%d"),
                 cl.JaroWinklerAtThresholds("city", [0.9]).configure(term_frequency_adjustments=True),
                 cl.EmailComparison("email")],
    blocking_rules_to_generate_predictions=[])            # no blocking rule: Splink scores all 499,500 pairs
for name in ("splink", "splink.internals"):                # Splink prints every training step: show only errors
    logging.getLogger(name).setLevel(logging.ERROR)
t0 = time.perf_counter()
p_linker = Linker(DuckDBAPI(connection=splink_connection()).register(raw.drop(columns="cluster").replace("", None)),
                  p_settings)
p_linker.training.estimate_probability_two_random_records_match([block_on("first_name", "surname", "dob")], recall=0.6)
p_linker.training.estimate_u_using_random_sampling(max_pairs=1e6, seed=1)
p_linker.training.estimate_parameters_using_expectation_maximisation(block_on("first_name", "surname"))
p_linker.training.estimate_parameters_using_expectation_maximisation(block_on("dob"))
scored = p_linker.inference.predict().as_pandas_dataframe()
p_splink_seconds = time.perf_counter() - t0
ids_l, ids_r = scored.unique_id_l.astype(int), scored.unique_id_r.astype(int)
w = dict(zip(zip(np.minimum(ids_l, ids_r), np.maximum(ids_l, ids_r)), scored.match_weight))   # (lower ID, higher ID)
p_vec["splink_weight"] = [w[(a, b)] for a, b in zip(p_pairs.id_l, p_pairs.id_r)]
t = best_threshold(p_vec.splink_weight, p_pairs)
print(f"trained without labels and scored all pairs in {p_splink_seconds:.1f} s; threshold {t:.2f}:",
      add_result("people", "Splink, no labels", p_vec.splink_weight >= t, p_pairs, t))

Logistic regression and LightGBM learn from the training pairs (Module 6). Every threshold is the one with the best F1 on validation.

In [ ]:
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

P_FIELDS = ["first_jw", "last_jw", "swapped_jw", "dob_lev", "dob_log_days", "city_jw", "email_jw"]
train = (p_pairs.split == "train").to_numpy()
y = p_pairs.same.to_numpy()

def with_missing(X):
    """For logistic regression: a missing value becomes -1, plus one column that says "missing"."""
    X = np.asarray(X, float)
    return np.column_stack([np.nan_to_num(X, nan=-1.0), np.isnan(X)])

def fit_lgbm(X, y):
    return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10,
                              random_state=1, n_jobs=1, verbose=-1).fit(X, y)

t0 = time.perf_counter()
logreg = LogisticRegression(max_iter=3000).fit(with_missing(p_vec[P_FIELDS])[train], y[train])
p_vec["logreg"] = logreg.predict_proba(with_missing(p_vec[P_FIELDS]))[:, 1]
p_lgbm = fit_lgbm(p_vec[P_FIELDS][train], y[train])
p_vec["lgbm"] = p_lgbm.predict_proba(p_vec[P_FIELDS])[:, 1]
p_learn_seconds = time.perf_counter() - t0
for name, col in [("logistic regression", "logreg"), ("LightGBM", "lgbm")]:
    t = best_threshold(p_vec[col], p_pairs)
    print(f"{name}: threshold {t:.4f},", add_result("people", name, p_vec[col] >= t, p_pairs, t))
print(f"trained on {train.sum():,} pairs ({y[train].sum():,} true) in {p_learn_seconds:.1f} s")
print(pd.DataFrame(results["people"]).T[["precision", "recall", "f1", "called", "right"]].to_string())

> **Check.** LightGBM wins: F1 0.956. Splink without labels (0.91) is better than logistic regression (0.88) here.

In [ ]:
gain = pd.Series(p_lgbm.booster_.feature_importance("gain"), index=P_FIELDS)
print((gain / gain.sum()).round(3).sort_values(ascending=False).to_string())

> **Check.** The email and the date of birth give most of the gain. The names count less, because the errors are mostly in the names.

## 6. Decide: accept, review or no match

The team accepts a merge without a person only when at least 99 in 100 such merges are right (an assumption). Pairs from 0.05 to the upper threshold go to a person.

In [ ]:
P_LOWER = 0.05                     # our choice, as in the course: below it, almost no pair is a match
p_grid = band_table(p_vec.lgbm, p_pairs, "validation", [0.5, 0.8, 0.95, 0.99], P_LOWER)
print(p_grid.to_string())
P_UPPER = min(up for up, r in p_grid.iterrows() if r["precision accepted"] >= 0.99)
p_decision = pd.concat({w: band_table(p_vec.lgbm, p_pairs, w, [P_UPPER], P_LOWER) for w in ["validation", "test"]}).droplevel(1)
print("upper threshold:", P_UPPER)
print(p_decision.to_string())

> **Check.** Upper threshold 0.99. On test, 440 merges are accepted (440 right) and 84 pairs go to a person.

> **Your turn.** Change one thing: train LightGBM without `swapped_jw`, and measure it on test. Predict first: does one field of 1.4% gain matter?

In [ ]:
# Your turn: train LightGBM again without the field "swapped_jw", choose its threshold on validation,
# and measure it on test.
fields_no_swap = [f for f in P_FIELDS if f != "swapped_jw"]
model_no_swap = ...      # fit_lgbm() on the training pairs, with fields_no_swap only
score_no_swap = ...      # its probability for every pair
t_no_swap = ...          # best_threshold() of that score
no_swap = ...            # pair_measures() on the test split
print(no_swap)
no_swap_f1 = no_swap["f1"] if isinstance(no_swap, dict) else None

In [ ]:
expect('no_swap_f1', no_swap_f1, 0.936, tolerance=0.001)

> **Check.** The F1 falls. A field with little gain can still decide the few hard pairs where the names were swapped.

## 7. From pairs to people

Connected components (Module 7) turn the links into clusters: one cluster per person.

In [ ]:
P_THRESHOLD = results["people"]["LightGBM"]["threshold"]
links = p_pairs[p_vec.lgbm >= P_THRESHOLD]
p_cluster = components(people.id, links.id_l, links.id_r)
p_clusters = cluster_measures(people, p_cluster)
print(f"{len(links):,} links ->", p_clusters)

### Failure case: a rule that worked on Larkfield

Module 7's safe clustering cut a link when two dates of birth disagreed. On these records:

In [ ]:
# Module 7's safe clustering cuts a link when two dates of birth disagree. Does that rule fit this list?
other_dob = (L.dob[p_pairs.same] != R.dob[p_pairs.same]).mean()
dobs = people.groupby("entity").dob.nunique()
several = people.entity.value_counts()
several = several[several > 1].index
print(f"true pairs with two different dates of birth: {other_dob:.0%}")
print(f"people with 2 or more records: {len(several)}; of them, with 2 or more dates of birth: {(dobs[several] > 1).sum()}")
strict = links[(L.dob[links.index] == R.dob[links.index]).to_numpy()]
p_strict = cluster_measures(people, components(people.id, strict.id_l, strict.id_r))
print("links that keep only the same date of birth:", p_strict)

> **Warning.** More than half of the true pairs have two different dates of birth, because the simulation makes typing errors in dates too. The rule keeps every cluster pure (B-cubed precision 1.0), but it breaks most people into pieces. Check a rule on the new data before you reuse it.

## 8. Golden records, and the design on new people

One merge rule per field: the most common non-empty value.

In [ ]:
def golden(records):
    """One merge rule per field (Module 7): the most common non-empty value; a tie goes to the lowest record ID."""
    out = {}
    for col in ["first", "last", "dob", "city", "email"]:
        values = records.sort_values("id")[col]
        values = values[values != ""]
        out[col] = values.value_counts(sort=True).index[0] if len(values) else ""
    return out

people["cluster"] = people.id.map(p_cluster)
one = people[people.cluster == people.cluster[people.id == 0].iloc[0]]
print(one[["id", "first", "last", "dob", "city", "email"]].to_string(index=False))
print("golden record:", golden(one))
p_golden = {c: golden(g) for c, g in people.groupby("cluster")}
print(f"{len(people)} records -> {len(p_golden)} golden records (true people: {people.entity.nunique()})")

`find_duplicate_people()` runs every step on a new list. You run it on the test people only, as if they were a new list.

In [ ]:
def find_duplicate_people(records, model, threshold):
    """The whole design on a new list: normalise, compare all pairs, score, link, cluster, golden records."""
    r = records.assign(first=records.first_name.map(text), last=records.surname.map(text),
                       city=records.city.map(text), email=records.email.str.strip().str.lower(),
                       id=records.unique_id.astype(int)).reset_index(drop=True)
    a, b = np.triu_indices(len(r), k=1)
    A, B = r.iloc[a].reset_index(drop=True), r.iloc[b].reset_index(drop=True)
    X = pd.DataFrame({"first_jw": sim(A["first"], B["first"]), "last_jw": sim(A["last"], B["last"]),
                      "swapped_jw": np.fmax(sim(A["first"], B["last"]), sim(A["last"], B["first"])),
                      "dob_lev": sim(A.dob, B.dob, Levenshtein.normalized_similarity),
                      "dob_log_days": np.log1p((pd.to_datetime(A.dob) - pd.to_datetime(B.dob)).dt.days.abs().to_numpy()),
                      "city_jw": sim(A.city, B.city), "email_jw": sim(A.email, B.email)})
    keep = model.predict_proba(X[P_FIELDS])[:, 1] >= threshold
    cl = components(r.id, A.id[keep], B.id[keep])
    r["cluster"] = r.id.map(cl)
    return r, {c: golden(g) for c, g in r.groupby("cluster")}

new_list = raw[people.split.eq("test").to_numpy()]                # the test people: no step has used them
t0 = time.perf_counter()
new_records, new_golden = find_duplicate_people(new_list, p_lgbm, P_THRESHOLD)
p_new_seconds = time.perf_counter() - t0
new_records["entity"] = new_list.cluster.astype(int).to_numpy()
p_new = cluster_measures(new_records.assign(split="test"), dict(zip(new_records.id, new_records.cluster)))
print(f"{len(new_list)} records of new people, {p_new_seconds:.1f} s -> {len(new_golden)} golden records:", p_new)

> **Check.** About as many golden records as true people, with B-cubed precision and recall near 0.98.

## Part 2. Companies

## 9. The situation

A bank reports its trades with each client's **LEI**, the legal entity identifier: a 20-character code for one legal entity. One company should have one LEI. Sometimes a company gets two, for example from two different LEI issuers. GLEIF, the foundation that runs the global LEI system, marks the second one **DUPLICATE** and names the LEI that stays.

The data is real: every LEI record whose legal address is in Ireland, from GLEIF's public API, under CC0. The labels are GLEIF's own DUPLICATE links. The notebook never uses the status or the link as an input.

In [ ]:
gleif = pd.read_csv("data/gleif-ie.csv.gz", dtype=str, keep_default_na=False)
forms = pd.read_csv("data/elf-codes.csv", dtype=str, keep_default_na=False)
print(gleif.shape)
print(gleif.columns.tolist())
print(gleif.registration_status.value_counts().to_dict())

In [ ]:
print("repeated LEIs:", int(gleif.lei.duplicated().sum()))
dup = gleif[gleif.duplicate_of != ""]
print("records that GLEIF marks DUPLICATE:", len(dup), "| their surviving LEI is in the file:", int(dup.duplicate_of.isin(gleif.lei).sum()))
print("categories:", gleif.category.value_counts().to_dict())
print("empty values:", {c: int((gleif[c] == "").sum()) for c in ["legal_form", "address", "postcode", "registration_number", "other_names"]})
pc = gleif.postcode[gleif.postcode != ""].value_counts()
print("most common postcode:", pc.index[0], "with", pc.iloc[0], "records")
print("most common legal forms:", gleif.legal_form.map(dict(zip(forms.legal_form, forms.name))).value_counts().head(4).to_dict())

> **Check.** 38,650 records, 180 marked DUPLICATE. Many records share an address: fund administrators register thousands of funds at one office.

In [ ]:
by_lei = gleif.set_index("lei")
show = ["legal_name", "legal_form", "address", "postcode", "registration_number"]
for lei in ["984500A57BF3C6557B66", "2549002X1FJY0YFFS256", "213800S38W27JLP3LO42"]:   # three GLEIF duplicates
    a = by_lei.loc[lei]
    print(a[show].to_dict())
    print("   GLEIF: duplicate of ->", by_lei.loc[a.duplicate_of][show].to_dict(), "\n")

## 10. Normalise company names

`company()` writes every legal form in one short form ("public limited company" and "p.l.c." become `plc`). `core()` drops the legal-form words. `sub_fund()` keeps the part after the last dash, because an umbrella fund writes "UMBRELLA PLC - SUB-FUND".

In [ ]:
FORMS = [("public limited company", "plc"), ("p l c", "plc"), ("designated activity company", "dac"),
         ("irish collective asset management vehicle", "icav"), ("unlimited company", "uc"),
         ("limited", "ltd"), ("teoranta", "ltd"), ("company", "co")]
FORM_WORDS = {"plc", "dac", "icav", "uc", "ltd", "co", "the"}

def company(s):
    """A company name: text(), and every way of writing a legal form in one short form (limited -> ltd)."""
    s = " " + text(s) + " "
    for long, short in FORMS:
        s = s.replace(" " + long + " ", " " + short + " ")
    return s.strip()

def core(name):
    """The name without its legal-form words: 'avoca clo 42 dac' -> 'avoca clo 42'."""
    return " ".join(w for w in name.split() if w not in FORM_WORDS)

def sub_fund(legal_name):
    """An umbrella fund writes 'UMBRELLA PLC - SUB-FUND': keep the part after the last dash."""
    parts = re.split(r"\s[-\u2013]\s", legal_name)
    return core(company(parts[-1]))

NO_NUMBER = {"", "n a", "na", "none", "not applicable"}
co = pd.DataFrame({"lei": gleif.lei, "name": gleif.legal_name.map(company)})
co["core"] = co.name.map(core)
co["sub"] = gleif.legal_name.map(sub_fund)
co["other"] = gleif.other_names.map(lambda s: [core(company(x)) for x in s.split(" | ") if x.strip()])
co["form"] = gleif.legal_form.where(~gleif.legal_form.isin(["8888", "9999"]), "")
co["category"] = gleif.category
co["address"] = gleif.address.map(text)
co["postcode"] = gleif.postcode.str.replace(" ", "").str.upper()
co["city"] = gleif.city.map(text)
co["reg"] = np.where(gleif.registration_number.map(text).isin(NO_NUMBER), "",
                     gleif.registration_authority + "|" + gleif.registration_number.str.upper().str.replace(" ", ""))
co["lou"] = gleif.managing_lou
co["year"] = pd.to_numeric(gleif.first_registered.str[:4], errors="coerce")

# The labels: GLEIF's own DUPLICATE links. Records joined by a chain of links are one company.
row = {lei: k for k, lei in enumerate(co.lei)}
link_l = [row[a] for a, b in zip(gleif.lei, gleif.duplicate_of) if b]
link_r = [row[b] for a, b in zip(gleif.lei, gleif.duplicate_of) if b]
co["entity"] = co.index.map(components(co.index, link_l, link_r))
co["id"] = co.index
c_split = split_entities(co.entity)
co["split"] = co.entity.map(c_split)
sizes = co.entity.value_counts()
c_true_pairs = int((sizes * (sizes - 1) // 2).sum())
print(f"{len(co):,} records; companies with 2 or more LEIs: {(sizes > 1).sum()}; true pairs: {c_true_pairs}")
print(co.loc[co.lei == "984500A57BF3C6557B66", ["name", "core", "sub", "form", "postcode", "reg"]].to_string(index=False))

> **Predict.** "Same name" sounds safe for companies. What precision do you expect for exact names?

In [ ]:
def same_value_pairs(col, cap=50):
    """All pairs that share one non-empty value; a value shared by more than `cap` records is skipped."""
    out = set()
    for value, rows in co.groupby(col).id:
        if value != "" and 1 < len(rows) <= cap:
            out |= set(itertools.combinations(sorted(rows), 2))
    return out

import itertools
true_set = {p for _, rows in co.groupby("entity").id if len(rows) > 1 for p in itertools.combinations(sorted(rows), 2)}

def as_pairs(pair_set):
    l, r = (np.array(x, dtype=int) for x in zip(*sorted(pair_set))) if pair_set else (np.array([], int), np.array([], int))
    pairs = pd.DataFrame({"id_l": l, "id_r": r})
    pairs["same"] = [(a, b) in true_set for a, b in zip(l, r)]
    pairs["split"] = pair_split(co.split.to_numpy()[l], co.split.to_numpy()[r])
    pairs.attrs["true_pairs"] = pd.Series([pair_split([co.split[a]], [co.split[b]])[0] for a, b in true_set]).value_counts().to_dict()
    return pairs

for name, col in [("exact name", "name"), ("rule: same registration number", "reg")]:
    found = as_pairs(same_value_pairs(col))
    print(f"{name}:", add_result("companies", name, np.ones(len(found), bool), found))

> **Check.** Precision 0.198: four in five "same name" pairs are not GLEIF duplicates. Many are sub-funds with the same name in two different umbrellas. The registration number is shared too: all sub-funds of one umbrella have the umbrella's number.

## 11. Blocking

All pairs of 38,656 records are about 747 million. Five blockings together keep the candidates small.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def tfidf_top(texts, k=10, chunk=500):
    """For every name, the k most similar other names (TF-IDF of 3-letter pieces, cosine)."""
    X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3), min_df=2).fit_transform(texts)
    out = set()
    for s in range(0, X.shape[0], chunk):
        S = (X[s:s + chunk] @ X.T).toarray()
        S[np.arange(S.shape[0]), np.arange(s, s + S.shape[0])] = -1      # not the record itself
        top = np.argpartition(-S, k, axis=1)[:, :k]
        out |= {(min(s + r, c), max(s + r, c)) for r in range(S.shape[0]) for c in top[r]}
    return out

def other_name_pairs():
    """A record's other name (often a previous legal name) is another record's name."""
    by_core = co.groupby("core").id.apply(list).to_dict()
    return {(min(i, j), max(i, j)) for i, names in zip(co.id, co.other) for n in names
            for j in by_core.get(n, [])[:50] if j != i}

t0 = time.perf_counter()
blocks = {"same name": same_value_pairs("core"), "same sub-fund name": same_value_pairs("sub"),
          "same registration number": same_value_pairs("reg", cap=20), "other name = name": other_name_pairs(),
          "TF-IDF top 10": tfidf_top(co.core)}
blocks["all five together"] = set().union(*blocks.values())
c_block_seconds = time.perf_counter() - t0
all_pairs = len(co) * (len(co) - 1) // 2
c_blocking = pd.DataFrame({name: {"candidates": len(b), "true kept": len(b & true_set),
                                  "pair completeness": round(len(b & true_set) / len(true_set), 3),
                                  "reduction ratio": round(1 - len(b) / all_pairs, 5)} for name, b in blocks.items()}).T
print(f"{all_pairs:,} possible pairs; blocking took {c_block_seconds:.0f} s")
print(c_blocking.to_string())
c_pairs = as_pairs(blocks["all five together"])

In [ ]:
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

def text_sim(a, b, scorer):
    return np.array([scorer(x, y) / (100 if scorer is fuzz.token_set_ratio else 1) if x and y else np.nan
                     for x, y in zip(a, b)])

def same(a, b):
    a, b = np.asarray(a, dtype=object), np.asarray(b, dtype=object)
    return np.where((a == "") | (b == ""), np.nan, (a == b).astype(float))

def other_name_sim(names_a, core_b):
    """The best similarity between one record's other names and the other record's name (missing if none)."""
    return np.array([max(fuzz.token_set_ratio(n, c) for n in ns) / 100 if ns and c else np.nan
                     for ns, c in zip(names_a, core_b)])

def company_vectors(pairs):
    A, B = co.loc[pairs.id_l].reset_index(drop=True), co.loc[pairs.id_r].reset_index(drop=True)
    return pd.DataFrame({
        "name_tsr": text_sim(A.core, B.core, fuzz.token_set_ratio),
        "name_jw": text_sim(A.core, B.core, JaroWinkler.normalized_similarity),
        "sub_tsr": text_sim(A["sub"], B["sub"], fuzz.token_set_ratio),
        "other_name": np.fmax(other_name_sim(A.other, B.core), other_name_sim(B.other, A.core)),
        "same_form": same(A.form, B.form),
        "same_category": same(A.category, B.category),
        "address_tsr": text_sim(A.address, B.address, fuzz.token_set_ratio),
        "same_postcode": same(A.postcode, B.postcode),
        "same_reg": same(A.reg, B.reg),
        "same_lou": same(A.lou, B.lou),
        "years_apart": (A.year - B.year).abs().to_numpy(dtype=float),
    })

t0 = time.perf_counter()
c_vec = company_vectors(c_pairs)
c_compare_seconds = time.perf_counter() - t0
C_FIELDS = list(c_vec.columns)
print(f"{len(c_vec):,} comparison vectors in {c_compare_seconds:.0f} s")
print(c_vec[c_pairs.same.to_numpy()].describe().loc[["mean"]].round(2).to_string())
print(c_vec[~c_pairs.same.to_numpy()].describe().loc[["mean"]].round(2).to_string())

## 12. Splink and the classifiers

In [ ]:
c_table = co.assign(unique_id=co.id)[["unique_id", "core", "sub", "form", "postcode", "address", "reg"]].replace("", None)
c_settings = SettingsCreator(
    link_type="dedupe_only", unique_id_column_name="unique_id",
    comparisons=[cl.JaroWinklerAtThresholds("core", [0.95, 0.88]).configure(term_frequency_adjustments=True),
                 cl.JaroWinklerAtThresholds("sub", [0.95]),
                 cl.ExactMatch("form"), cl.ExactMatch("postcode"),
                 cl.JaroWinklerAtThresholds("address", [0.9]), cl.ExactMatch("reg")],
    blocking_rules_to_generate_predictions=[block_on("core"), block_on("sub"), block_on("reg"),
                                            block_on("substr(core, 1, 10)", "postcode")])
t0 = time.perf_counter()
c_linker = Linker(DuckDBAPI(connection=splink_connection()).register(c_table), c_settings)
c_linker.training.estimate_probability_two_random_records_match([block_on("core", "postcode")], recall=0.5)
c_linker.training.estimate_u_using_random_sampling(max_pairs=5e6, seed=1)
c_linker.training.estimate_parameters_using_expectation_maximisation(block_on("core"))
c_linker.training.estimate_parameters_using_expectation_maximisation(block_on("postcode"))
scored = c_linker.inference.predict().as_pandas_dataframe()
c_splink_seconds = time.perf_counter() - t0
ids_l, ids_r = scored.unique_id_l.astype(int), scored.unique_id_r.astype(int)
w = dict(zip(zip(np.minimum(ids_l, ids_r), np.maximum(ids_l, ids_r)), scored.match_weight))
c_vec["splink_weight"] = [w.get((a, b), -99.0) for a, b in zip(c_pairs.id_l, c_pairs.id_r)]   # -99: outside Splink's blocking
t = best_threshold(c_vec.splink_weight, c_pairs)
print(f"Splink trained without labels in {c_splink_seconds:.1f} s; threshold {t:.2f}:",
      add_result("companies", "Splink, no labels", c_vec.splink_weight >= t, c_pairs, t))

Only 76 true pairs are in the training split. So the LightGBM here is **small**: 4 leaves, and at least 30 pairs in each leaf.

In [ ]:
def fit_small_lgbm(X, y):
    """A small LightGBM for few labels: 4 leaves, at least 30 pairs per leaf, and a penalty on large weights."""
    return lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, num_leaves=4, min_child_samples=30,
                              reg_lambda=5.0, random_state=1, n_jobs=1, verbose=-1).fit(X, y)

c_train = (c_pairs.split == "train").to_numpy()
c_y = c_pairs.same.to_numpy()
t0 = time.perf_counter()
c_logreg = LogisticRegression(max_iter=3000, class_weight="balanced").fit(with_missing(c_vec[C_FIELDS])[c_train], c_y[c_train])
c_vec["logreg"] = c_logreg.predict_proba(with_missing(c_vec[C_FIELDS]))[:, 1]
c_lgbm = fit_small_lgbm(c_vec[C_FIELDS][c_train], c_y[c_train])
c_vec["lgbm"] = c_lgbm.predict_proba(c_vec[C_FIELDS])[:, 1]
c_learn_seconds = time.perf_counter() - t0
for name, col in [("logistic regression", "logreg"), ("LightGBM (small)", "lgbm")]:
    t = best_threshold(c_vec[col], c_pairs)
    print(f"{name}: threshold {t:.4f},", add_result("companies", name, c_vec[col] >= t, c_pairs, t))
print(f"trained on {c_train.sum():,} pairs ({c_y[c_train].sum()} true) in {c_learn_seconds:.1f} s")
print(pd.DataFrame(results["companies"]).T[["precision", "recall", "f1", "called", "right", "true"]].to_string())

### Failure case: a big model on few labels

The people's LightGBM (15 leaves, 300 trees) on the same training pairs:

In [ ]:
# The people's LightGBM (15 leaves, 300 trees) on the companies' 76 true training pairs:
big = fit_lgbm(c_vec[C_FIELDS][c_train], c_y[c_train]).predict_proba(c_vec[C_FIELDS])[:, 1]
t = best_threshold(big, c_pairs)
c_big = {"validation": pair_measures(big >= t, c_pairs, "validation"), "test": pair_measures(big >= t, c_pairs, "test")}
print(f"threshold {t:.4f}", c_big)
print("share of training pairs that the big model scores above 0.99:", round(float((big[c_train] > 0.99).mean()), 5),
      "| true share:", round(float(c_y[c_train].mean()), 5))

> **Warning.** With few true pairs, a big model learns the training pairs by heart. Its F1 on test falls to 0.103. With few labels, choose a small model, and let the validation split decide.

In [ ]:
c_gain = pd.Series(c_lgbm.booster_.feature_importance("gain"), index=C_FIELDS)
print((c_gain / c_gain.sum()).round(3).sort_values(ascending=False).to_string())

## 13. Are the "wrong" matches wrong?

In [ ]:
# Look at the wrong matches of the small LightGBM on test: are they really two companies?
C_THRESHOLD = results["companies"]["LightGBM (small)"]["threshold"]
called = (c_vec.lgbm >= C_THRESHOLD).to_numpy() & (c_pairs.split == "test").to_numpy()
wrong = c_pairs[called & ~c_y]
A, B = co.loc[wrong.id_l].reset_index(drop=True), co.loc[wrong.id_r].reset_index(drop=True)
# Our own rule, not GLEIF's: same name (after normalise) and the same registration number -> probably one company.
probably_same = ((A.core == B.core) & (A.reg == B.reg) & (A.reg != "")).to_numpy()
maybe_same = ((A.core == B.core) & ((A.reg == "") | (B.reg == ""))).to_numpy()
c_unflagged = {"wrong matches": len(wrong), "same name and registration number": int(probably_same.sum()),
               "same name, one number missing": int(maybe_same.sum())}
print(c_unflagged)
cols = ["legal_name", "registration_status", "registration_number", "postcode", "first_registered"]
for a, b in list(zip(wrong.id_l, wrong.id_r))[:6]:
    print(gleif.loc[a, cols].tolist(), "\n   ", gleif.loc[b, cols].tolist())

> **Check.** Many of the "wrong" matches have the same name. Some also have the same registration number: they are probably one company that GLEIF has not marked. Others have the same name and one number is missing: they may be one company, or two sub-funds. A real register's labels are **incomplete**: the measured precision is lower than the true one. The rule "same name and same number" is our own label, not GLEIF's.

How much does one split decide? Ten other seeds for the split:

In [ ]:
# How much do the results depend on the split? Ten other seeds for the split, the same candidates and fields.
rows = {}
for seed in range(10):
    sp = co.entity.map(split_entities(co.entity, seed=seed)).to_numpy()
    pairs = c_pairs.assign(split=pair_split(sp[c_pairs.id_l], sp[c_pairs.id_r]))
    pairs.attrs["true_pairs"] = pd.Series([pair_split([sp[a]], [sp[b]])[0] for a, b in true_set]).value_counts().to_dict()
    tr = (pairs.split == "train").to_numpy()
    score = fit_small_lgbm(c_vec[C_FIELDS][tr], c_y[tr]).predict_proba(c_vec[C_FIELDS])[:, 1]
    A, B = co.loc[pairs.id_l].reset_index(drop=True), co.loc[pairs.id_r].reset_index(drop=True)
    rows[seed] = {"test true pairs": pairs.attrs["true_pairs"]["test"],
                  "exact name": pair_measures((A.name == B.name).to_numpy(), pairs)["f1"],
                  "LightGBM (small)": pair_measures(score >= best_threshold(score, pairs), pairs)["f1"]}
c_seeds = pd.DataFrame(rows).T
print(c_seeds.to_string())
print(c_seeds.describe().loc[["min", "50%", "max"]].round(3).to_string())

> **Check.** The test split has only 31 to 59 true pairs, so the F1 moves by about 0.2 from one seed to another. LightGBM beats the exact names on every seed. Do not read the third decimal.

## 14. Decide

In [ ]:
C_LOWER = 0.05                     # our choice: the lowest score worth a person's time (see the text)
c_grid = band_table(c_vec.lgbm, c_pairs, "validation", [0.5, 0.7, 0.9, 0.95], C_LOWER)
print(c_grid.to_string())

In [ ]:
TARGET = 0.99                      # the team's rule for a merge without a person (an assumption)
passing = [up for up, r in c_grid.iterrows() if r["precision accepted"] >= TARGET]
print("upper thresholds that meet the rule on validation:", passing or "none")
C_UPPER = 1.01                     # so no pair is merged without a person: every pair from C_LOWER up is reviewed
queue = {}
for w in ["validation", "test"]:
    ins = (c_pairs.split == w).to_numpy() & (c_vec.lgbm >= C_LOWER).to_numpy()
    A, B = co.loc[c_pairs.id_l[ins]].reset_index(drop=True), co.loc[c_pairs.id_r[ins]].reset_index(drop=True)
    likely = ((A.core == B.core) & (A.reg == B.reg) & (A.reg != "")).to_numpy() & ~c_y[ins]
    queue[w] = {"to review": int(ins.sum()), "GLEIF duplicates": int(c_y[ins].sum()),
                "probably the same, not marked (our rule)": int(likely.sum()),
                "recall if reviews are right": round(float(c_y[ins].sum() / c_pairs.attrs["true_pairs"][w]), 3)}
c_queue = pd.DataFrame(queue).T
print(c_queue.to_string())

Below 0.05, the reviewer would find almost nothing (our choice of the lower threshold).

> **Check.** No upper threshold reaches 99 in 100 on validation. So nothing merges without a person. On test, 255 pairs go to a person, and 30 of them are GLEIF duplicates.

> **Your turn.** The reviewer has time for 30 pairs a week. How many duplicates are in the first 30 pairs, sorted by score?

In [ ]:
# Your turn: the review queue has room for 30 pairs a week. Take the 30 test pairs with the highest LightGBM
# score (at least C_LOWER), and count how many are GLEIF duplicates.
test_band = c_pairs.assign(score=c_vec.lgbm.to_numpy())[(c_pairs.split == "test").to_numpy()]
test_band = ...          # keep the pairs with a score of at least C_LOWER
top30 = ...              # the 30 pairs with the highest score
found_in_30 = ...        # how many of them are true duplicates (the column `same`)
print(found_in_30, "duplicates in the first 30 reviews")

In [ ]:
expect('found_in_30', found_in_30, 18)

## 15. One LEI per company, and the design on new records

In [ ]:
# Cluster the reviewed pairs (as if the reviewer said yes to every GLEIF duplicate), and keep one LEI per company.
STATUS_ORDER = {"ISSUED": 0, "PENDING_TRANSFER": 1, "PENDING_ARCHIVAL": 2, "LAPSED": 3, "RETIRED": 4, "ANNULLED": 5, "DUPLICATE": 6}
kept = c_pairs[(c_vec.lgbm >= C_LOWER).to_numpy() & c_y]
c_cluster = components(co.id, kept.id_l, kept.id_r)
c_clusters = cluster_measures(co, c_cluster)
print(c_clusters)

def keep_lei(records):
    """The merge rule for the LEI (our choice): the best registration status, then the oldest LEI."""
    r = records.assign(rank=records.registration_status.map(STATUS_ORDER)).sort_values(["rank", "first_registered"])
    return r.lei.iloc[0]

gleif["cluster"] = gleif.index.map(c_cluster)
groups = gleif[gleif.cluster.map(gleif.cluster.value_counts()) > 1].groupby("cluster")
example = groups.get_group(c_cluster[row["984500A57BF3C6557B66"]])
print(example[["lei", "legal_name", "registration_status", "first_registered"]].to_string(index=False))
print("keep:", keep_lei(example))
agree = [keep_lei(g) == (g.duplicate_of[g.duplicate_of != ""].iloc[0] if (g.duplicate_of != "").any() else None) for _, g in groups]
print(f"groups with 2 or more LEIs: {len(agree)}; the kept LEI is GLEIF's surviving LEI in {sum(agree)}")

In [ ]:
def find_duplicate_companies(records, model, upper=C_UPPER, lower=C_LOWER):
    """The whole design on a new list of LEI records: normalise, block, compare, score, and three bands."""
    global co
    saved = co
    try:
        g = records.reset_index(drop=True)
        new = pd.DataFrame({"lei": g.lei, "name": g.legal_name.map(company)})
        new["core"], new["sub"] = new.name.map(core), g.legal_name.map(sub_fund)
        new["other"] = g.other_names.map(lambda s: [core(company(x)) for x in s.split(" | ") if x.strip()])
        new["form"] = g.legal_form.where(~g.legal_form.isin(["8888", "9999"]), "")
        new["category"], new["address"] = g.category, g.address.map(text)
        new["postcode"], new["city"] = g.postcode.str.replace(" ", "").str.upper(), g.city.map(text)
        new["reg"] = np.where(g.registration_number.map(text).isin(NO_NUMBER), "",
                              g.registration_authority + "|" + g.registration_number.str.upper().str.replace(" ", ""))
        new["lou"], new["year"] = g.managing_lou, pd.to_numeric(g.first_registered.str[:4], errors="coerce")
        new["id"] = new.index
        co = new
        cand = set().union(same_value_pairs("core"), same_value_pairs("sub"), same_value_pairs("reg", cap=20),
                           other_name_pairs(), tfidf_top(new.core))
        l, r = (np.array(x) for x in zip(*sorted(cand)))
        pairs = pd.DataFrame({"lei_l": new.lei[l].to_numpy(), "lei_r": new.lei[r].to_numpy(), "id_l": l, "id_r": r})
        score = model.predict_proba(company_vectors(pairs)[C_FIELDS])[:, 1]
    finally:
        co = saved
    pairs["score"] = score
    pairs["decision"] = np.select([score >= upper, score >= 0.5, score >= lower], ["merge", "review first", "review"], "no match")
    return pairs[pairs.decision != "no match"]

new_list = gleif[co.split.eq("test").to_numpy()]            # the test companies: no step has used them
t0 = time.perf_counter()
run = find_duplicate_companies(new_list, c_lgbm)
c_new_seconds = time.perf_counter() - t0
truth = set(zip(gleif.lei[link_l], gleif.lei[link_r])) | set(zip(gleif.lei[link_r], gleif.lei[link_l]))
ent = dict(zip(gleif.lei, co.entity))
run["same"] = [ent[a] == ent[b] for a, b in zip(run.lei_l, run.lei_r)]
c_new = run.groupby("decision").same.agg(["count", "sum"]).astype(int)
print(f"{len(new_list):,} new LEI records, {c_new_seconds:.0f} s; true duplicate pairs among them: "
      f"{pair_measures(np.zeros(len(c_pairs), bool), c_pairs, 'test')['true']}")
print(c_new.to_string())

> **Check.** On the new records, most "review first" pairs (score 0.5 or more) are GLEIF duplicates. The pairs below need a careful look.

## 16. Recap

- Simulated people: LightGBM on seven similarities found the duplicates with F1 near 0.96, and the merges above 0.99 were right on test.
- A rule from Larkfield (two dates of birth cannot be one person) failed on these records.
- Real companies: names and numbers are shared by many different funds. Exact names were wrong four times in five.
- With 76 true training pairs, a small LightGBM beat a big one.
- GLEIF's labels are incomplete. The design sends every pair to a person, sorted by score.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/record-matching/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/record-matching/m09/find-duplicate-people-and-companies/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The people are simulated (Splink's fake_1000, MIT). The companies are real LEI records from GLEIF (CC0).